In [0]:

dbutils.widgets.text("catalog", "ridepulse", "Catalog name")
dbutils.widgets.text("source_path", "/Volumes/ridepulse/landing/raw/hvfhv_stream/", "Source directory to watch")
dbutils.widgets.text("checkpoint_path", "/Volumes/ridepulse/landing/checkpoints/bronze_trips_stream/", "Checkpoint location")
dbutils.widgets.dropdown("trigger_mode", "availableNow", ["availableNow", "continuous"], "Trigger mode")

catalog = dbutils.widgets.get("catalog")
source_path = dbutils.widgets.get("source_path")
checkpoint_path = dbutils.widgets.get("checkpoint_path")
trigger_mode = dbutils.widgets.get("trigger_mode")

target_table = f"{catalog}.bronze.trips_stream"
schema_location = f"{checkpoint_path}_schema"

spark.conf.set("spark.sql.parquet.inferTimestampNTZ.enabled", "false")

from pyspark.sql.functions import col, current_timestamp, input_file_name

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)


query = stream_df.writeStream.format("delta").option("checkpointLocation", checkpoint_path)

if trigger_mode == "availableNow":
    query = query.trigger(availableNow=True)
else:
    query = query.trigger(processingTime="30 seconds")

streaming_query = query.toTable(target_table)
streaming_query.awaitTermination()


row_count = spark.table(target_table).count()
print(f"{target_table}: {row_count:,} rows")

display(spark.table(target_table).orderBy(col("ingestion_timestamp").desc()).limit(10))
